In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Punjabi Bagh, Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,Eth-Benzene,RH,WS,WD,SR,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,200.47,289.21,16.25,45.31,37.31,1.54,12.58,NaN,NaN,NaN,82.44,0.56,197.47,53.71,NaN,14.36,NaN,0
1,02-01-2025 00:00,03-01-2025 00:00,203.98,285.67,12.27,42.38,32.51,1.83,12.27,NaN,NaN,NaN,85.05,0.68,191.49,46.75,NaN,14.41,NaN,0
2,03-01-2025 00:00,04-01-2025 00:00,278.12,393.42,29.11,51.44,51.03,3.50,11.05,NaN,NaN,NaN,83.59,0.36,190.08,57.04,NaN,15.44,NaN,0
3,04-01-2025 00:00,05-01-2025 00:00,307.83,510.92,21.62,62.98,51.07,2.92,11.76,NaN,NaN,NaN,83.54,0.37,174.25,55.65,NaN,15.87,NaN,0
4,05-01-2025 00:00,06-01-2025 00:00,192.75,273.08,12.18,44.33,33.49,1.71,8.40,NaN,NaN,NaN,81.14,0.55,157.53,66.66,NaN,14.75,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,354.08,526.12,13.79,45.39,35.35,1.69,41.03,NaN,NaN,NaN,53.41,0.42,178.36,72.85,NaN,19.09,NaN,0
316,13-11-2025 00:00,14-11-2025 00:00,304.60,487.12,6.26,46.19,29.66,1.18,35.62,NaN,NaN,NaN,56.71,0.36,180.64,67.39,NaN,18.78,NaN,0
317,14-11-2025 00:00,15-11-2025 00:00,247.56,414.58,4.00,44.74,27.05,1.18,38.21,NaN,NaN,NaN,55.88,0.37,177.66,70.16,NaN,18.47,NaN,0
318,15-11-2025 00:00,16-11-2025 00:00,257.29,424.96,7.12,46.52,30.53,1.41,36.17,NaN,NaN,NaN,54.67,0.42,172.69,72.48,NaN,18.47,NaN,0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 18)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Benzene', 'Toluene', 'Eth-Benzene']
Dropped rows (>70% NaN): 6
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
CO           0
Ozone        0
RH           0
WS           0
WD           0
SR           0
AT           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")

In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (314, 15)
          From Date           To Date   PM2.5    PM10      NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  200.47  289.21  16.250  45.31  37.31   
1  02-01-2025 00:00  03-01-2025 00:00  203.98  285.67  12.270  42.38  32.51   
2  03-01-2025 00:00  04-01-2025 00:00   65.09  393.42   5.485  51.44  51.03   
3  04-01-2025 00:00  05-01-2025 00:00   65.09  152.21  21.620  62.98  51.07   
4  05-01-2025 00:00  06-01-2025 00:00  192.75  273.08  12.180  44.33  33.49   

      CO  Ozone     RH    WS      WD     SR     AT  TOT-RF  
0  1.540  12.58  82.44  0.56  197.47  53.71  14.36       0  
1  1.005  12.27  85.05  0.68  191.49  46.75  14.41       0  
2  1.005  11.05  83.59  0.36  190.08  57.04  15.44       0  
3  1.005  11.76  83.54  0.37  174.25  55.65  15.87       0  
4  1.710   8.40  81.14  0.55  157.53  66.66  14.75       0  


In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,RH,WS,WD,SR,AT,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,2.807722,1.438011,1.555640,0.807443,1.049375,2.048957,-0.995475,1.486580,0.089967,1.206907,-0.424406,-2.143492,0.0
1,02-01-2025 00:00,03-01-2025 00:00,2.884234,1.396694,0.878050,0.603625,0.669063,-0.140847,-1.012390,1.672257,0.653084,0.793513,-0.704235,-2.134793,0.0
2,03-01-2025 00:00,04-01-2025 00:00,-0.143356,2.654287,-0.277088,1.233862,2.136433,-0.140847,-1.078961,1.568392,-0.848561,0.696041,-0.290523,-1.955604,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.143356,-0.160971,2.469875,2.036614,2.139602,-0.140847,-1.040219,1.564835,-0.801635,-0.398278,-0.346408,-1.880797,0.0
4,05-01-2025 00:00,06-01-2025 00:00,2.639437,1.249751,0.862727,0.739272,0.746710,2.744783,-1.223562,1.394098,0.043041,-1.554122,0.096252,-2.075643,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
309,12-11-2025 00:00,13-11-2025 00:00,-0.143356,-0.160971,1.136828,0.813008,0.894081,2.662921,0.556933,-0.578628,-0.567003,-0.114156,0.345123,-1.320614,0.0
310,13-11-2025 00:00,14-11-2025 00:00,-0.143356,-0.160971,-0.145145,0.868658,0.443253,0.575444,0.261730,-0.343865,-0.848561,0.043459,0.125602,-1.374545,0.0
311,14-11-2025 00:00,15-11-2025 00:00,-0.143356,2.901254,-0.529907,0.767792,0.236458,0.575444,0.403056,-0.402911,-0.801635,-0.162547,0.236970,-1.428475,0.0
312,15-11-2025 00:00,16-11-2025 00:00,-0.143356,3.022403,0.001269,0.891614,0.512185,1.516855,0.291741,-0.488991,-0.567003,-0.506120,0.330247,-1.428475,0.0


In [10]:
df.to_excel('pungabibagh2025.xlsx', index=False)